<a href="https://colab.research.google.com/github/jetendarsoothar-png/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jetendarsoothar-png/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1 — Content age and health score

The paper reports that content health scores were higher for newer content and lower for older content, with the reported score declining across older content-age groups.

**My methodology question:** How was the health-score label or outcome defined and measured for each content item, and were other factors such as content type or traffic level considered when comparing content-age groups? I would want to understand whether the observed difference is an association rather than evidence that content age itself caused the decline.

## Finding 2 — CTR and search position

The paper reports different weighted CTR levels across search-position groups, with higher CTR for higher-ranking positions.

**My methodology question:** Is this result an observed aggregate relationship, or was it also validated across separate clients or time periods? If the validation is based mainly on aggregate comparisons, I would describe the result as a measured directional association rather than a causal effect of search position on CTR.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## My model under an honest split

In Week 5, the decision tree reported Precision@50 of 0.720. That result was based on the earlier evaluation setup.

For this audit, I used a grouped client split so that pages from the same client were not placed in both training and testing data. The split used 25 clients for training and 7 unseen clients for testing.

The grouped test set had a declining-label base rate of 0.680. On the unseen clients, the model measured Precision@20 of 0.950 and Precision@50 of 0.840.

Compared with the Week-5 reported Precision@50 of 0.720, the grouped test Precision@50 was 0.120 higher in this run. This is an observed difference in this evaluation, not evidence that the model will always perform better on unseen clients.

The grouped result gives more useful decision-support evidence because the test clients were not used for training.

In [4]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/jetendarsoothar-png/flyrank-ml-internship.git"
REPO_DIR = "flyrank-ml-internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

os.chdir(REPO_DIR)

print("Now in:", os.getcwd())
print("Dataset exists:", os.path.exists(
    "data/raw/content_refresh_anonymized.csv"
))

Now in: /content/flyrank-ml-internship
Dataset exists: True


In [5]:
# Section 2 — Honest validation: grouped by client

import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# Load the starter dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create the same label used in Week 5
df["is_declining_label"] = (
    df["trend_direction"].str.lower().eq("down").astype(int)
)

# Same pre-decision features used in Week 5
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = (
    df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = df["is_declining_label"].values
groups = df["client_id"].values


# --------------------------------------------------
# 1. Before: Week-5 reported result
# --------------------------------------------------

week5_precision_50 = 0.720

print("Week-5 reported Precision@50:", week5_precision_50)


# --------------------------------------------------
# 2. Honest grouped split
# --------------------------------------------------

unique_clients = np.sort(df["client_id"].unique())

# Keep the split reproducible
rng = np.random.RandomState(42)
rng.shuffle(unique_clients)

split_point = int(len(unique_clients) * 0.80)

train_clients = unique_clients[:split_point]
test_clients = unique_clients[split_point:]

train_mask = df["client_id"].isin(train_clients)
test_mask = df["client_id"].isin(test_clients)

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]

y_train = y[train_mask]
y_test = y[test_mask]


# --------------------------------------------------
# 3. Train only on training clients
# --------------------------------------------------

tree = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42
)

tree.fit(X_train, y_train)


# --------------------------------------------------
# 4. Predict only on unseen test clients
# --------------------------------------------------

test_scores = tree.predict_proba(X_test)[:, 1]


def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()


# --------------------------------------------------
# 5. Honest Precision@20 and Precision@50
# --------------------------------------------------

print("\nGrouped client split results:")
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Training rows:", len(y_train))
print("Testing rows:", len(y_test))

print("\nTest base rate:", round(y_test.mean(), 3))

for k in (20, 50):
    score = precision_at_k(test_scores, y_test, k)
    print(f"Grouped Precision@{k}: {score:.3f}")


# --------------------------------------------------
# 6. Before vs After comparison
# --------------------------------------------------

grouped_precision_50 = precision_at_k(test_scores, y_test, 50)

print("\nBefore vs After")
print(f"Week-5 reported Precision@50: {week5_precision_50:.3f}")
print(f"Grouped test Precision@50:    {grouped_precision_50:.3f}")
print(f"Difference:                   {grouped_precision_50 - week5_precision_50:+.3f}")

Week-5 reported Precision@50: 0.72

Grouped client split results:
Training clients: 25
Testing clients: 7
Training rows: 24220
Testing rows: 5780

Test base rate: 0.68
Grouped Precision@20: 0.950
Grouped Precision@50: 0.840

Before vs After
Week-5 reported Precision@50: 0.720
Grouped test Precision@50:    0.840
Difference:                   +0.120


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Leakage audit

The final feature set contains six pre-decision signals: content age, days since last update, impressions, average position, CTR, and word count.

The label `is_declining_label` is derived from `trend_direction`, while `trend_pct` is the numeric signal used to determine that direction. Therefore, both `trend_direction` and `trend_pct` were explicitly excluded from the feature set.

The audit confirms that neither label-derived column is present in the final features. This reduces the risk of direct label leakage.

The current features are based on page/content state and trailing-90-day measurements. Their timing should still be checked against the exact label window before treating the evaluation as fully leakage-free.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Final Week-5 feature set
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

# Known label-derived / suspicious columns
suspect_features = [
    "trend_direction",
    "trend_pct"
]

print("Final features:")
for feature in features:
    print(" -", feature)

print("\nLeakage check:")
for feature in suspect_features:
    print(f"{feature}: {'LEAKED - NOT USED' if feature not in features else 'WARNING - USED'}")

# Check for direct overlap with label construction
label_source = "trend_direction"
label_numeric_source = "trend_pct"

print("\nLabel:")
print("is_declining_label = trend_direction == 'down'")

print("\nResult:")
print("trend_direction in features:", label_source in features)
print("trend_pct in features:", label_numeric_source in features)

# Simple timeline note for the current starter features
print("\nTimeline audit:")
print("The current features describe page/content state and trailing-90-day measurements.")
print("The label is derived from trend_direction.")
print("Therefore trend_direction and trend_pct must remain excluded from features.")

Final features:
 - content_age_days
 - days_since_last_update
 - impressions_90d
 - avg_position
 - ctr
 - word_count

Leakage check:
trend_direction: LEAKED - NOT USED
trend_pct: LEAKED - NOT USED

Label:
is_declining_label = trend_direction == 'down'

Result:
trend_direction in features: False
trend_pct in features: False

Timeline audit:
The current features describe page/content state and trailing-90-day measurements.
The label is derived from trend_direction.
Therefore trend_direction and trend_pct must remain excluded from features.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Claim rewrite

### Original claim

The decision tree outperformed the hand rule.

### Safer claim

In the Week-5 evaluation, the decision tree measured higher Precision@50 than the hand rule (0.720 vs. 0.680), while the hand rule measured higher Precision@20 (0.900 vs. 0.700). Under the grouped client split used in this audit, the decision tree measured Precision@50 of 0.840 on 7 unseen clients.

These results are observed and measured differences in the evaluated datasets. They provide directional decision-support evidence, but they do not establish that the decision tree will always outperform the hand rule or that the model causes pages to decline.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.